**AI Company Brochure Generator**

In this project, am building an AI tool that creates a short company brochure from a company’s website.

The tool:

Scrapes the company’s main website.
Uses an LLM to identify useful pages, such as About, Careers, and Company pages.
Collects content from those relevant pages.
Uses an LLM to generate a polished Markdown brochure for potential customers, investors, and job candidates.
Streams the result so the brochure appears gradually, like ChatGPT typing.

Goal
1.import the tools need
2.set up and verfiy the API connection 
3.collect the company websites main content
4.collect all website links 
5.build a prompt asking openai to choose the most useful links 
6.ask ai to select the relevant links 
7.convert the ais JSONs answer into a python list 
8.scrape the selected pages 
9.combine the website information into one context package 
10.build the brochure-writing prompt 
11.Ask the ai to write the brochure
12.stream and display the brochure in the notebook



In [1]:
import os                                                         #used to interact with my computer environment. going to use it to read out api key from environment variable 
import json                                                       #import the built in json module - ot turn json text to python 
from dotenv import load_dotenv                                    #using this function to load secret values from .env file
from IPython.display import Markdown, display, update_display     #tools spexifically useful inside jupyter notebook ie. turning tex into formatted markdown
from scraper import fetch_website_links, fetch_website_contents   #my custom module 
from openai import OpenAI                                         #importing the open ai class to create client object, which we use to send requests to an openai model

In [ ]:
# Setting up and verifying the openai api connection 
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key?")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
#pulling function from scraper.py module. this function pulls the links from a webpage 
links = fetch_website_links("https://marvodyn.com/")
links

['#content',
 '#mobile-menu',
 'https://marvodyn.com',
 '#',
 'https://marvodyn.com/',
 'https://marvodyn.com/credit/',
 'https://marvodyn.com/banking/',
 'https://marvodyn.com/investing/',
 'https://marvodyn.com/taxes/',
 'https://marvodyn.com/budgeting-saving/',
 'https://marvodyn.com/loans/',
 'https://marvodyn.com/students/',
 'https://marvodyn.com/gm/',
 'https://marvodyn.com/tools/',
 'https://marvodyn.com/resources/',
 'https://marvodyn.com/shop/',
 'https://marvodyn.com/about/',
 '#',
 'javascript:;',
 'javascript:;',
 'javascript:;',
 'javascript:;',
 'javascript:;',
 'javascript:;',
 'javascript:;',
 'https://marvodyn.com/welcome__trashed/welcome-page-try/',
 'https://marvodyn.com/credit-in-america-explained-for-immigrants-start-here/',
 'https://marvodyn.com/credit-in-america-explained-for-immigrants-start-here/',
 'https://marvodyn.com/category/credit/',
 'https://marvodyn.com/banking-in-america-explained-for-immigrants-start-here/',
 'https://marvodyn.com/banking-in-americ

In [ ]:
#using gpt 5 nano to figure out which links are relevant 

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
#instead of rewriting everything all the time, we create a function that takes a website url and builds the prompt that asks the ai which link are useful for a company brochure
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://marvodyn.com/"))


Here is the list of links on the website https://marvodyn.com/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#content
#mobile-menu
https://marvodyn.com
#
https://marvodyn.com/
https://marvodyn.com/credit/
https://marvodyn.com/banking/
https://marvodyn.com/investing/
https://marvodyn.com/taxes/
https://marvodyn.com/budgeting-saving/
https://marvodyn.com/loans/
https://marvodyn.com/students/
https://marvodyn.com/gm/
https://marvodyn.com/tools/
https://marvodyn.com/resources/
https://marvodyn.com/shop/
https://marvodyn.com/about/
#
javascript:;
javascript:;
javascript:;
javascript:;
javascript:;
javascript:;
javascript:;
https://marvodyn.com/welcome__trashed/welcome-page-try/
https://marvodyn.com/credit-in-america-explained-for-immigrants-start-here/
https://marvodyn.com/credit-in-america-explained-for-im

In [7]:
#function that gives the al all the links from a website, then gets back only the links useful for making a company brouchure 
def select_relevant_links(url):
    response = openai.chat.completions.create(                         #sends the request to the ai model
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},         #gives the ai its permanent instructions, such as what kinds of links count as relevant
            {"role": "user", "content": get_links_user_prompt(url)}    #gives the ai the actual website links for this specific company
        ],
        response_format={"type": "json_object"}                        #forces the ai to return json instead of normal writing
    )
    result = response.choices[0].message.content                       #get the ai answer as text 
    links = json.loads(result)                                         #converts the json text into a ptyhon dictionary
    return links                                                       #sends the cleaned, selected links back to wherever you called the function
    

In [8]:
select_relevant_links("https://marvodyn.com/")

{'links': [{'type': 'homepage', 'url': 'https://marvodyn.com/'},
  {'type': 'about page', 'url': 'https://marvodyn.com/about/'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/marvodyn/about/?viewAsMember=true'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/profile.php?id=61560758706657'},
  {'type': 'pinterest page', 'url': 'https://www.pinterest.com/marvodyn/'},
  {'type': 'instagram profile', 'url': 'https://www.instagram.com/marvodyn/'},
  {'type': 'twitter/x profile', 'url': 'https://x.com/marvodyn'}]}

In [9]:
#making the brochure 
#function 
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result


In [10]:
print(fetch_page_and_all_relevant_links("https://marvodyn.com/"))

## Landing Page:

marvodyn.com

Skip to content
Menu
Search
Home
Credit
Banking
Investing
Taxes
Budgeting & Saving
Loans
Students
Global Money
Tools
Resources
Shop
About
Close Menu
•
•
•
•
•
←
→
MARVODYN
Money Made Simple for Immigrants in America
Press Here to Start
BEGINNER GUIDES
March 6, 2026
Credit in America Explained for Immigrants: A Complete Beginner Guide
Credit
Why Credit Matters More Than You Might Think You made a big decision coming to the United States. You may…
March 6, 2026
Banking in America Explained for Immigrants: A Complete Beginner Guide
Banking
A System That Feels Unfamiliar at First When we arrive in the United States, one of the first practical challenges…
March 6, 2026
Taxes in America Explained for Immigrants: A Beginner’s Guide
Taxes
A System That Surprises Almost Every New Arrival When we come to the United States, taxes are rarely at the…
March 6, 2026
Financial Guide for International Students in the United States (Complete Beginner Guide)
Students
The F

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [12]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [16]:
create_brochure("marvodyn", "https://marvodyn.com/")

# marvodyn Brochure

---

## About marvodyn

**Money Made Simple for Immigrants in America**  
marvodyn is a dedicated financial platform designed specifically for immigrants navigating the complex American financial system. Understanding that moving to the United States is a major life decision filled with challenges and uncertainties, marvodyn aims to simplify money management from banking and credit to investing, loans, and taxes.

Every year, millions of newcomers arrive in America with ambition and determination but face financial systems that are confusing and unfamiliar. marvodyn provides clear, beginner-friendly guides and tailored financial advice to help immigrants confidently take their first steps towards financial stability and growth.

---

## Our Mission

To empower immigrants in America by delivering straightforward, trustworthy financial education and practical resources — making it easier to understand credit scores, apply for bank accounts, navigate taxes, and invest wisely.

---

## Key Features & Services

### Extensive Beginner Guides
- **Credit in America Explained:** Why credit matters and how to build it.
- **Banking Basics:** Opening bank accounts, understanding ITINs, and navigating services without a Social Security Number.
- **Taxes Simplified:** A beginner’s guide to American tax systems for new arrivals.
- **Financial Tips for International Students:** Managing finances while studying in the U.S.

### Curated Financial Options
- Best banks for ITIN holders (no SSN required)
- Top personal and auto loans for immigrants, including no-credit options
- Best high-yield savings accounts and no-fee checking accounts
- Beginner-friendly investment apps and brokerage accounts starting from $1
- Recommended money transfer apps and services optimized for immigrants sending money internationally

---

## Who We Serve

- **Immigrants at all stages** learning to navigate finances in the U.S.
- **International students** adjusting to the American financial landscape
- **Workers building credit and savings** with limited or no SSN
- Anyone seeking **financial clarity and reliable resources** designed with immigrant experiences in mind

---

## Company Culture

marvodyn is committed to inclusivity, empowerment, and education. The company embraces diversity, ensuring its tools and advice are culturally sensitive and accessible. By listening closely to immigrant communities, marvodyn fosters a supportive culture aimed at removing barriers and boosting confidence in personal finance.

---

## Careers & Opportunities

marvodyn encourages passionate individuals to join their mission of financial inclusion. Opportunities may be available in content creation, financial education, technology, and customer support. Ideal candidates are empathetic, detail-oriented, and eager to make a meaningful social impact.

---

## Connect & Learn More

Visit [marvodyn.com](https://marvodyn.com) to explore resources, guides, and tools crafted to help immigrant communities succeed financially in the United States.

---

**marvodyn** — Making money matters simple, one immigrant at a time.